In [7]:
import pandas as pd
import numpy as np

In [8]:
path_file = 'Embrapii_seleção_analista_2026_questao01_Eventos.xlsx'

try:
    print("Iniciando a leitura do arquivo Excel...")
    xls = pd.read_excel(path_file, sheet_name=None)
    
    df_raw = pd.concat(xls.values(), ignore_index=True)
    
    print(f"Sucesso! {len(df_raw)} registros carregados de todas as abas.")
    display(df_raw.head(3))

except FileNotFoundError:
    print(f"ERRO: O arquivo '{path_file}' não foi encontrado. Verifique se o nome está correto e se ele está na mesma pasta do seu Jupyter Notebook.")
except Exception as e:
    print(f"ERRO INESPERADO ao carregar os dados: {e}")

Iniciando a leitura do arquivo Excel...
Sucesso! 369 registros carregados de todas as abas.


,Data início,Data final,Formato,Cidade,Embrapii Day,Evento,Tipo,Organizador,Responsável EMBRAPII,Temas,Programas/Iniciativas envolvidas,Observação
0,2023-01-09 00:00:00,NaT,Online,NaN,Sim,EMBRAPII Day Online CSN Inova,Próprio,EMBRAPII,Pessoa 01,Bio/Economia Circular/Biocombustíveis/Bioecono...,Geral,NaN
1,2023-01-17 00:00:00,NaT,Online,NaN,Sim,EMBRAPII Day Online Schaeffler,Próprio,EMBRAPII,Pessoa 01,Automotivo,Rota 2030,NaN
2,2023-01-18 00:00:00,NaT,Híbrido,São Paulo,Não,CCET 50 anos,Externo,UFSCAR,Pessoa 02,Geral,Geral,Palestra Menezes ''Perspectivas para os financ...


In [9]:
try:
    colunas_padrao = {
        'Evento': 'Nome_Evento',
        'Data início': 'Data_Inicio',
        'Data final': 'Data_Final',
        'Formato': 'Formato',
        'Cidade': 'Local_Cidade',
        'Embrapii Day': 'Flag_Embrapii_Day',
        'Tipo': 'Tipo_Organizacao',
        'Organizador': 'Organizador',
        'Responsável EMBRAPII': 'Responsavel_Embrapii',
        'Temas': 'Temas_Abordados',
        'Programas/Iniciativas envolvidas': 'Programas_Iniciativas',
        'Observação': 'Observacoes'
    }

    df_cleaned = df_raw.rename(columns=colunas_padrao)

    colunas_necessarias = ['ID_Evento', 'Nome_Evento', 'Data_Inicio', 'Data_Final', 'Formato', 
                           'Local_Cidade', 'Tipo_Organizacao', 'Organizador', 'Flag_Embrapii_Day', 
                           'Responsavel_Embrapii', 'Temas_Abordados', 'Programas_Iniciativas', 'Observacoes']

    for col in colunas_necessarias:
        if col not in df_cleaned.columns and col != 'ID_Evento':
            df_cleaned[col] = np.nan
            
    print("Colunas padronizadas com sucesso.")

except KeyError as e:
    print(f"ERRO DE CHAVE: A coluna esperada não foi encontrada no dataframe original: {e}")
except Exception as e:
    print(f"ERRO INESPERADO ao padronizar colunas: {e}")

Colunas padronizadas com sucesso.


In [10]:
try:
    df_cleaned['Data_Final'] = df_cleaned['Data_Final'].fillna(df_cleaned['Data_Inicio'])
    
    df_cleaned['Data_Inicio'] = pd.to_datetime(df_cleaned['Data_Inicio'], errors='coerce').dt.normalize()
    df_cleaned['Data_Final'] = pd.to_datetime(df_cleaned['Data_Final'], errors='coerce').dt.normalize()

    mask_embrapii = df_cleaned['Nome_Evento'].str.contains('Embrapii Day', case=False, na=False)
    df_cleaned.loc[mask_embrapii, 'Flag_Embrapii_Day'] = 'Sim'
    df_cleaned['Flag_Embrapii_Day'] = df_cleaned['Flag_Embrapii_Day'].fillna('Não')

    mask_online = df_cleaned['Formato'].str.contains('Online', case=False, na=False)
    df_cleaned.loc[mask_online & df_cleaned['Local_Cidade'].isna(), 'Local_Cidade'] = 'Não aplicável (Online)'

    cols_texto = ['Local_Cidade', 'Organizador', 'Responsavel_Embrapii', 'Temas_Abordados', 'Programas_Iniciativas', 'Observacoes']
    for c in cols_texto:
        df_cleaned[c] = df_cleaned[c].fillna('Não informado')
        
    print("Limpeza e regras de negócio aplicadas com sucesso.")

except TypeError as e:
    print(f"ERRO DE TIPO: Falha ao aplicar função em uma coluna de tipo incorreto: {e}")
except Exception as e:
    print(f"ERRO INESPERADO durante a limpeza de dados: {e}")

Limpeza e regras de negócio aplicadas com sucesso.


In [11]:
try:
    df_cleaned = df_cleaned.sort_values(by='Data_Inicio').reset_index(drop=True)

    ids_evento = ['EVT-' + str(i).zfill(4) for i in range(1, len(df_cleaned) + 1)]
    
    if 'ID_Evento' not in df_cleaned.columns or df_cleaned['ID_Evento'].isnull().all():
        df_cleaned['ID_Evento'] = ids_evento
    
    df_final = df_cleaned[colunas_necessarias]

    print("Estruturação finalizada com sucesso! Prévia dos dados:")
    display(df_final.head())

except ValueError as e:
    print(f"ERRO DE VALOR: Falha na criação dos IDs ou ordenação: {e}")
except Exception as e:
    print(f"ERRO INESPERADO na estruturação final: {e}")

Estruturação finalizada com sucesso! Prévia dos dados:


,ID_Evento,Nome_Evento,Data_Inicio,Data_Final,Formato,Local_Cidade,Tipo_Organizacao,Organizador,Flag_Embrapii_Day,Responsavel_Embrapii,Temas_Abordados,Programas_Iniciativas,Observacoes
0,EVT-0001,EMBRAPII Day Online CSN Inova,2023-01-09,2023-01-09,Online,Não aplicável (Online),Próprio,EMBRAPII,Sim,Pessoa 01,Bio/Economia Circular/Biocombustíveis/Bioecono...,Geral,Não informado
1,EVT-0002,EMBRAPII Day Online Schaeffler,2023-01-17,2023-01-17,Online,Não aplicável (Online),Próprio,EMBRAPII,Sim,Pessoa 01,Automotivo,Rota 2030,Não informado
2,EVT-0003,CCET 50 anos,2023-01-18,2023-01-18,Híbrido,São Paulo,Externo,UFSCAR,Não,Pessoa 02,Geral,Geral,Palestra Menezes ''Perspectivas para os financ...
3,EVT-0004,ANFAVEA + Hyundai | Rota 2030,2023-01-19,2023-01-19,Presencial,São Paulo,Externo,ANFAVEA,Não,Pessoa 03,Automotivo,Rota 2030,Não informado
4,EVT-0005,Apresentação EMBRAPII | APLA - Arranjo Produti...,2023-01-20,2023-01-20,Presencial,São Paulo,Externo,APLA,Não,Pessoa 03,Bio/Economia Circular/Biocombustíveis/Bioecono...,BNDES,Não informado


In [12]:
try:
    output_filename = 'Eventos_Estruturados_Final.xlsx'
    df_final.to_excel(output_filename, index=False)
    
    print(f"Sucesso! Arquivo '{output_filename}' exportado corretamente.")

except PermissionError:
    print(f"ERRO DE PERMISSÃO: O arquivo '{output_filename}' parece estar aberto em outro programa. Feche o Excel e tente rodar esta célula novamente.")
except Exception as e:
    print(f"ERRO INESPERADO ao exportar o arquivo: {e}")

Sucesso! Arquivo 'Eventos_Estruturados_Final.xlsx' exportado corretamente.
